# Python en GPU NVIDIA — notebook ejecutable

Acompaña al artículo **«Python en GPU NVIDIA: Activar Full Aceleración»** de [stivenson.github.io](https://stivenson.github.io/#/articles/python-en-gpu-nvidia).

Aquí está **todo el código del artículo**, listo para ejecutar de arriba a abajo. Cada sección mide en tu propia sesión lo que el artículo explica en teoría.

## Antes de empezar

1. Menú **Entorno de ejecución → Cambiar tipo de entorno de ejecución → Acelerador por hardware: GPU (T4)**.
2. Ejecuta las celdas en orden. La sección 4 (`cuml.accel`) instala cuML y **reinicia el kernel**: después de ese reinicio, sigue desde la celda que lo indica.

> **Los tiempos que obtengas serán distintos a los del artículo.** Esa es justamente la idea: el artículo mide en un portátil AMD Ryzen 7 5700U sin GPU; aquí mides en la GPU que te asigne Colab. Compara *tus* dos columnas, no las mías.

## 0. Qué hardware te tocó

Colab reparte distintas GPU (T4, L4, A100…). Todo lo que midas depende de cuál te haya tocado, así que empieza por anotarla.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import platform, subprocess
print("CPU:", subprocess.run(["bash","-lc","grep -m1 'model name' /proc/cpuinfo | cut -d: -f2"], capture_output=True, text=True).stdout.strip())
print("Núcleos lógicos:", __import__("os").cpu_count())
print("Python:", platform.python_version())

## 1. Antes de la GPU: el problema suele ser Python

Replica de la tabla de Leiserson et al. (*Science*, 2020) en pequeño: la misma multiplicación de matrices, con tres bucles `for` de Python y con NumPy.

In [ ]:
import time, random
import numpy as np

n = 256
A = [[random.random() for _ in range(n)] for _ in range(n)]
B = [[random.random() for _ in range(n)] for _ in range(n)]

# --- Python puro: tres bucles anidados ---
t0 = time.perf_counter()
C = [[0.0] * n for _ in range(n)]
for i in range(n):
    for j in range(n):
        for k in range(n):
            C[i][j] += A[i][k] * B[k][j]
t_python = time.perf_counter() - t0

# --- NumPy: la misma operación, en código compilado ---
a, b = np.array(A), np.array(B)
tiempos = []
for _ in range(20):
    t0 = time.perf_counter()
    c = a @ b
    tiempos.append(time.perf_counter() - t0)
t_numpy = sorted(tiempos)[len(tiempos) // 2]   # mediana

print(f"Python puro : {t_python:8.3f} s")
print(f"NumPy       : {t_numpy*1000:8.3f} ms")
print(f"Diferencia  : {t_python/t_numpy:8,.0f}x")
print(f"Mismo resultado: diferencia máxima {np.abs(np.array(C) - c).max():.2e}")

## 2. El costo de mover los datos (Gregg & Hazelwood, 2011)

El artículo estima este costo con las velocidades de catálogo. Aquí lo **mides**: cuánto tarda de verdad un arreglo en viajar por el PCIe, y cuánto tarda la GPU en leerlo una vez que ya está en su memoria.

In [ ]:
import cupy as cp
import numpy as np
import time

def cronometrar(fn, repeticiones=5):
    """Mide una operación de GPU correctamente: descarta la primera
    ejecución (compilación y arranque) y sincroniza antes de parar el reloj."""
    fn()                                   # calentamiento, se descarta
    cp.cuda.Device().synchronize()
    tiempos = []
    for _ in range(repeticiones):
        t0 = time.perf_counter()
        fn()
        cp.cuda.Device().synchronize()     # esperar a que la GPU termine DE VERDAD
        tiempos.append(time.perf_counter() - t0)
    return sorted(tiempos)[len(tiempos) // 2]

n_bytes = 512 * 1024 * 1024                # 512 MB
host = np.ones(n_bytes // 4, dtype=np.float32)
dev = cp.asarray(host)

t_subida = cronometrar(lambda: cp.asarray(host))
t_bajada = cronometrar(lambda: cp.asnumpy(dev))
t_suma = cronometrar(lambda: dev.sum())

gb = n_bytes / 1e9
print(f"RAM  -> VRAM : {t_subida*1000:7.1f} ms   ({gb/t_subida:6.1f} GB/s)  <- esto es el PCIe")
print(f"VRAM -> RAM  : {t_bajada*1000:7.1f} ms   ({gb/t_bajada:6.1f} GB/s)")
print(f"Sumar en GPU : {t_suma*1000:7.1f} ms   ({gb/t_suma:6.1f} GB/s)  <- esto es la memoria interna")
print()
print(f"Mover los datos cuesta {t_subida/t_suma:.0f} veces más que sumarlos.")
print("Conclusión: la GPU gana cuando los datos SE QUEDAN en ella y hacen mucho trabajo.")

### Intensidad aritmética: por qué unas operaciones aprovechan la GPU y otras no

Dos operaciones sobre datos del mismo tamaño, con intensidades aritméticas muy distintas (modelo *roofline*, Williams et al., 2009).

In [ ]:
import cupy as cp

n = 4096
x = cp.random.random((n, n), dtype=cp.float32)
y = cp.random.random((n, n), dtype=cp.float32)

t_suma = cronometrar(lambda: x + y)          # ~0,08 operaciones por byte
t_matmul = cronometrar(lambda: x @ y)        # ~680 operaciones por byte

flops_suma = n * n                           # una suma por elemento
flops_matmul = 2 * n ** 3                    # multiplicación de matrices

print(f"Suma elemento a elemento : {t_suma*1000:7.2f} ms  ->  {flops_suma/t_suma/1e12:6.3f} TFLOP/s")
print(f"Multiplicación de matrices: {t_matmul*1000:7.2f} ms  ->  {flops_matmul/t_matmul/1e12:6.3f} TFLOP/s")
print()
print("La suma mueve muchos datos y calcula poco: la GPU pasa el tiempo esperando memoria.")
print("El producto de matrices reutiliza cada dato cientos de veces: ahí la GPU trabaja a tope.")

### La trampa del `float64` en tarjetas de consumo

Las GPU de centro de datos (A100, H100) son rápidas en `float64`. Las de consumo (T4, L4, RTX) son mucho más lentas. NumPy y pandas usan `float64` **por defecto**.

In [ ]:
import cupy as cp

n = 4096
for dtype in (cp.float32, cp.float64):
    a = cp.random.random((n, n), dtype=dtype)
    t = cronometrar(lambda: a @ a)
    print(f"{str(dtype.__name__):>8}: {t*1000:8.2f} ms  ->  {2*n**3/t/1e12:6.3f} TFLOP/s")

## 3. pandas en GPU sin cambiar el código: `cudf.pandas`

En Colab con GPU, cuDF ya viene instalado. La extensión se carga **antes** de importar pandas.

In [ ]:
%load_ext cudf.pandas
import pandas as pd
print("pandas:", pd.__version__, "| módulo real:", type(pd.DataFrame).__module__)

In [ ]:
import numpy as np, pandas as pd, time

filas = 20_000_000
rng = np.random.default_rng(0)
df = pd.DataFrame({
    "grupo": rng.integers(0, 1000, filas),
    "valor": rng.random(filas),
    "peso": rng.random(filas),
})

t0 = time.perf_counter()
resumen = (df.assign(ponderado=df["valor"] * df["peso"])
             .groupby("grupo")
             .agg(media=("ponderado", "mean"), total=("peso", "sum"), n=("valor", "size"))
             .sort_values("media", ascending=False))
t = time.perf_counter() - t0

print(f"groupby + agg sobre {filas:,} filas: {t*1000:.1f} ms")
resumen.head()

### Ver qué corrió en la GPU y qué no

El paso que el artículo llama obligatorio. `%%cudf.pandas.profile` resume por función; `%%cudf.pandas.line_profile`, línea por línea.

In [ ]:
%%cudf.pandas.profile
resumen = (df.assign(ponderado=df["valor"] * df["peso"])
             .groupby("grupo")
             .agg(media=("ponderado", "mean"), total=("peso", "sum")))

In [ ]:
%%cudf.pandas.line_profile
filtrado = df[df["valor"] > 0.5]
por_grupo = filtrado.groupby("grupo")["peso"].sum()
texto = df["grupo"].astype(str).str.zfill(4)     # las operaciones de texto suelen ir a la CPU

### La misma comparación, contra pandas de verdad

Para comparar con honestidad hay que ejecutar el mismo trabajo con pandas **sin acelerar**. Se hace en un proceso aparte, porque en este la extensión ya está cargada.

Ojo con el detalle que repite el artículo: el tiempo de la GPU debe incluir **construir el dataframe**, que es cuando los datos viajan.

In [ ]:
%%writefile /content/bench_pandas.py
import json, os, time, sys
import numpy as np, pandas as pd

filas = 20_000_000
rng = np.random.default_rng(0)

t0 = time.perf_counter()
df = pd.DataFrame({
    "grupo": rng.integers(0, 1000, filas),
    "valor": rng.random(filas),
    "peso": rng.random(filas),
})
t_crear = time.perf_counter() - t0

t0 = time.perf_counter()
resumen = (df.assign(ponderado=df["valor"] * df["peso"])
             .groupby("grupo")
             .agg(media=("ponderado", "mean"), total=("peso", "sum"), n=("valor", "size")))
t_calcular = time.perf_counter() - t0

etiqueta = sys.argv[1] if len(sys.argv) > 1 else "pandas"
print(f"{etiqueta:>18} | crear: {t_crear*1000:8.1f} ms | calcular: {t_calcular*1000:8.1f} ms | total: {(t_crear+t_calcular)*1000:8.1f} ms")

# Se guarda para el gráfico final: así la figura la dibujan TUS números.
os.makedirs("/content/resultados", exist_ok=True)
with open(f"/content/resultados/pandas_{etiqueta.split()[0]}.json", "w") as f:
    json.dump({"etiqueta": etiqueta, "crear_ms": t_crear*1000, "calcular_ms": t_calcular*1000}, f)

In [ ]:
!python /content/bench_pandas.py "pandasCPU (CPU)"
!python -m cudf.pandas /content/bench_pandas.py "cudfGPU (GPU)"

## 4. scikit-learn en GPU: `cuml.accel`

Esta celda instala cuML y **reinicia el kernel**. Es normal: Colab avisa de que la sesión se reinició. Después del reinicio, continúa desde la celda siguiente (no hace falta repetir las anteriores).

In [ ]:
!pip install -q --extra-index-url=https://pypi.nvidia.com "cuml-cu12>=25.2"
import IPython
IPython.Application.instance().kernel.do_shutdown(True)   # reinicia el kernel

### Continúa aquí después del reinicio

Primero, el script del artículo tal cual, **en CPU**, en sus dos versiones: la de por defecto (un solo núcleo) y la que usa todos los núcleos. La diferencia entre ambas es la que decide si tu «speedup» son 14× o 2,9×.

In [ ]:
%%writefile /content/sklearn_workflow.py
import json, os, time, sys
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

n_jobs = None if len(sys.argv) < 2 or sys.argv[1] == "default" else -1

t0 = time.time()
X, y = make_classification(n_classes=2, n_features=10, n_samples=100_000, random_state=0)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0)
t_datos = time.time() - t0

model = RandomForestClassifier(max_depth=10, n_estimators=25, n_jobs=n_jobs, random_state=0)
t0 = time.time(); model.fit(X_train, y_train); t_fit = time.time() - t0
t0 = time.time(); pred = model.predict(X_test); t_pred = time.time() - t0

exactitud = accuracy_score(y_test, pred)
print(f"datos: {t_datos*1000:7.1f} ms | fit: {t_fit*1000:8.1f} ms | predict: {t_pred*1000:7.1f} ms | exactitud: {exactitud:.4f}")

# El nombre del archivo lo pone quien llama: sirve para el gráfico final.
etiqueta = os.environ.get("ETIQUETA", "sin_nombre")
os.makedirs("/content/resultados", exist_ok=True)
with open(f"/content/resultados/sklearn_{etiqueta}.json", "w") as f:
    json.dump({"etiqueta": etiqueta, "datos_ms": t_datos*1000, "fit_ms": t_fit*1000,
               "predict_ms": t_pred*1000, "exactitud": exactitud}, f)

In [ ]:
print("CPU, un núcleo (el valor por defecto de scikit-learn):")
!ETIQUETA=cpu_1nucleo python /content/sklearn_workflow.py default
print("\nCPU, todos los núcleos (n_jobs=-1):")
!ETIQUETA=cpu_todos python /content/sklearn_workflow.py paralelo
print("\nGPU con cuml.accel, sin tocar el código:")
!ETIQUETA=gpu python -m cuml.accel /content/sklearn_workflow.py default

### El perfil línea a línea de la charla

La columna **GPU %** dice qué parte del tiempo de cada línea corrió en la GPU. Es la diapositiva que el artículo analiza con la ley de Amdahl: mira cuánto del tiempo total está en las líneas aceleradas y cuánto en importar librerías y preparar datos.

In [ ]:
!python -m cuml.accel --line-profile /content/sklearn_workflow.py default

In [ ]:
!python -m cuml.accel --profile /content/sklearn_workflow.py default

### Cuando el modelo se devuelve a la CPU sin avisar

`sample_weight` es una de las condiciones que hacen que `RandomForestClassifier` no se pueda acelerar. Con `-v`, cuML dice por qué.

In [ ]:
%%writefile /content/fallback_demo.py
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification

X, y = make_classification(n_samples=50_000, n_features=10, random_state=0)
pesos = np.random.default_rng(0).random(len(y))
RandomForestClassifier(n_estimators=25, max_depth=10, random_state=0).fit(X, y, sample_weight=pesos)
print("entrenado")

In [ ]:
!python -m cuml.accel -v /content/fallback_demo.py

## 5. Cuando las librerías no alcanzan: CuPy y Numba

Los dos fragmentos que el artículo muestra como el escalón siguiente.

In [ ]:
import cupy as cp

x = cp.random.random((10_000, 10_000), dtype=cp.float32)
y = x @ x.T                                  # se ejecuta en la GPU
resultado = cp.asnumpy(y.sum(axis=0))        # traer a la CPU SOLO lo necesario

print(type(y), y.shape, y.dtype)
print("Primeros valores ya en CPU:", resultado[:3])

In [ ]:
import numpy as np
from numba import cuda

@cuda.jit
def sumar_uno(arr):
    i = cuda.grid(1)          # cada hilo sabe qué posición le toca
    if i < arr.size:
        arr[i] += 1

datos = np.zeros(1_000_000, dtype=np.float32)
arr_en_gpu = cuda.to_device(datos)

# 1 millón de elementos -> ~4.000 bloques de 256 hilos cada uno
sumar_uno[(arr_en_gpu.size + 255) // 256, 256](arr_en_gpu)
cuda.synchronize()

print("Todos valen 1:", bool((arr_en_gpu.copy_to_host() == 1).all()))

## 6. El gráfico: con y sin optimización

Esta celda lee las mediciones que fueron guardando las secciones anteriores y dibuja la comparación. **Son tus números, en tu GPU.**

Si alguna barra falta, es que no ejecutaste esa sección; vuelve atrás y repítela.

In [ ]:
import json, glob, os
import matplotlib.pyplot as plt

# Paleta categórica validada (contraste y visión con deficiencia de color).
AZUL, NARANJA, VERDE = "#2a78d6", "#eb6834", "#1baf7a"

def ms(v):
    """Formato español: punto para los miles, coma para los decimales."""
    return f"{v:,.1f} ms".replace(",", "·").replace(".", ",").replace("·", ".")

def cargar(patron):
    datos = {}
    for ruta in sorted(glob.glob(f"/content/resultados/{patron}")):
        with open(ruta) as f:
            d = json.load(f)
        datos[os.path.basename(ruta).split(".")[0]] = d
    return datos

sk = cargar("sklearn_*.json")
pd_ = cargar("pandas_*.json")

orden_sk = [("sklearn_cpu_1nucleo", "CPU, 1 núcleo", AZUL),
            ("sklearn_cpu_todos",   "CPU, todos",    NARANJA),
            ("sklearn_gpu",         "GPU",           VERDE)]
orden_pd = [("pandas_pandasCPU", "pandas (CPU)",      AZUL),
            ("pandas_cudfGPU",   "cudf.pandas (GPU)", VERDE)]

paneles = [
    ("Entrenar el modelo", [(e, c, sk[k]["fit_ms"]) for k, e, c in orden_sk if k in sk]),
    ("Predecir",           [(e, c, sk[k]["predict_ms"]) for k, e, c in orden_sk if k in sk]),
    ("groupby sobre 20M de filas", [(e, c, pd_[k]["calcular_ms"]) for k, e, c in orden_pd if k in pd_]),
]
paneles = [(t, filas) for t, filas in paneles if filas]

fig, ejes = plt.subplots(len(paneles), 1, figsize=(8, 2.1 * len(paneles)))
ejes = [ejes] if len(paneles) == 1 else list(ejes)

for ax, (titulo, filas) in zip(ejes, paneles):
    etiquetas = [f[0] for f in filas]
    colores = [f[1] for f in filas]
    valores = [f[2] for f in filas]
    barras = ax.barh(etiquetas, valores, color=colores, height=0.55)
    # Cada escala es propia: entrenar tarda mucho más que predecir, y
    # compartir eje dejaría el panel corto en una línea invisible.
    ax.set_title(f"{titulo}  ·  menos es mejor (ms)", fontsize=10.5, loc="left")
    ax.invert_yaxis()
    ax.set_xlim(0, max(valores) * 1.28)
    for barra, v in zip(barras, valores):
        # El valor va escrito: nadie tiene que medir la barra con la vista.
        ax.text(v * 1.03, barra.get_y() + barra.get_height() / 2,
                ms(v), va="center", fontsize=9.5)
    for lado in ("top", "right"):
        ax.spines[lado].set_visible(False)
    ax.tick_params(labelsize=9.5)

fig.suptitle("Con y sin aceleración, en esta sesión", fontsize=13, y=1.0, x=0.01, ha="left")
fig.tight_layout()
plt.show()

# La misma información en texto, por si el gráfico no se ve bien.
for titulo, filas in paneles:
    print(f"\n{titulo}")
    base = filas[0][2]
    for etiqueta, _, v in filas:
        print(f"  {etiqueta:<22} {ms(v):>12}   {base/v:5.2f}x frente a {filas[0][0]}")

## 7. Tu propia tabla de resultados

Anota aquí lo que te dio a ti. Es el resumen que deberías publicar si cuentas estos números a alguien: **siempre con el hardware y contra qué versión de CPU se comparan**.

| Medición | Tu resultado |
|---|---|
| GPU asignada por Colab | |
| Python puro vs NumPy (sección 1) | |
| PCIe vs memoria interna (sección 2) | |
| `float32` vs `float64` (sección 2) | |
| pandas vs `cudf.pandas` (sección 3) | |
| `fit` en CPU 1 núcleo / CPU todos / GPU (sección 4) | |
| Fracción del script que corrió en GPU (sección 4) | |

Y el recordatorio del artículo: un resultado de 3× contra un rival bien optimizado vale más que uno de 100× contra un rival mal configurado.